In [2]:
!pip install mdtraj
import os
import numpy as np
import mdtraj as md
import cupy as cp
import matplotlib.pyplot as plt
from collections import OrderedDict

dna_color_map = {"A": "#d62728", "C": "#1f77ff", "G": "#2ca02c", "T": "#ff7e00"}

def gpu_contact_frequency(traj_chunk, cutoff, van_indices, dna_indices, van_map, dna_map, n_van, n_dna):
    coords = traj_chunk.xyz
    n_frames = coords.shape[0]
    van_coords = cp.asarray(coords[:, van_indices, :])
    dna_coords = cp.asarray(coords[:, dna_indices, :])
    batch_size = min(64, n_frames)
    contact_freq = cp.zeros((n_van, n_dna), dtype=cp.float32)

    for i in range(0, n_frames, batch_size):
        end = min(i + batch_size, n_frames)
        v = van_coords[i:end]
        d = dna_coords[i:end]
        dists = cp.linalg.norm(v[:, :, None, :] - d[:, None, :, :], axis=-1)
        contacts = dists < cutoff
        for frame_contacts in contacts:
            res_contacts = cp.zeros((n_van, n_dna), dtype=cp.float32)
            cp.maximum.at(res_contacts, (van_map[:, None], dna_map[None, :]), frame_contacts.astype(cp.float32))
            contact_freq += res_contacts

    freq = (contact_freq / n_frames).get()
    return np.clip(freq, 0, 1)

def generate_four_category_maps(contact_matrix, van_labels, dna_seq, dna_colors_regional, y_labels, y_colors, 
                               title_prefix, figsize, dpi, out_dir, original_cmap="Reds", original_vmin=0, original_vmax=1):
    
    categories = {
        'no_contact': {
            'name': 'No Probability Contact',
            'range': (0.0, 0.2),
            'color': original_cmap,
            'description': '0~0.2'
        },
        'lowest': {
            'name': 'Lowest Probability Contact', 
            'range': (0.2, 0.8),
            'color': original_cmap,
            'description': '0.2~0.8'
        },
        'highest': {
            'name': 'Highest Probability Contact',
            'range': (0.8, 1.0),
            'color': original_cmap,
            'description': '0.8~1'
        },
        'wrap': {
            'name': 'Wrap Contact (Full Range)',
            'range': (0.0, 1.0),
            'color': original_cmap,
            'description': '0~1 (Complete)'
        }
    }
    
    categories_data = {}
    categories_data['no_contact'] = np.where((contact_matrix >= 0.0) & (contact_matrix < 0.2), contact_matrix, 0.0)
    categories_data['lowest'] = np.where((contact_matrix >= 0.2) & (contact_matrix < 0.8), contact_matrix, 0.0)
    categories_data['highest'] = np.where((contact_matrix >= 0.8) & (contact_matrix <= 1.0), contact_matrix, 0.0)
    categories_data['wrap'] = contact_matrix.copy()
    
    average_results_dir = os.path.join(out_dir, "average_results")
    os.makedirs(average_results_dir, exist_ok=True)
    
    four_cat_dir = os.path.join(average_results_dir, "four_category_maps")
    total_avg_dir = os.path.join(average_results_dir, "total_average")
    os.makedirs(four_cat_dir, exist_ok=True)
    os.makedirs(total_avg_dir, exist_ok=True)
    
    print("\nGenerating Four Category Contact Maps in 'average_results' folder:")
    print("="*70)
    
    for category_key, category_info in categories.items():
        matrix = categories_data[category_key]
        rows, cols = matrix.shape
        
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        
        if category_key == 'wrap':
            vmin, vmax = original_vmin, original_vmax
        else:
            non_zero_values = matrix[matrix > 0]
            if len(non_zero_values) > 0:
                vmin = max(non_zero_values.min(), original_vmin)
                vmax = min(non_zero_values.max(), original_vmax)
            else:
                vmin, vmax = original_vmin, original_vmax
        
        y0, y1 = 0, rows
        im = ax.imshow(matrix, origin='lower', aspect='auto', 
                      extent=[0, cols, y0, y1], 
                      cmap=category_info['color'], vmin=vmin, vmax=vmax)

        ax.set_xlim(0, cols)
        x_ticks = np.arange(cols) + 0.5
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(list(dna_seq), fontsize=6, rotation=0, ha='center')
        for tick, color in zip(ax.get_xticklabels(), dna_colors_regional):
            tick.set_color(color)

        ax.set_ylim(y0, y1)
        y_ticks = np.arange(rows) + 0.5
        ax.set_yticks(y_ticks)
        ax.set_yticklabels(y_labels, fontsize=8)
        for tick, color in zip(ax.get_yticklabels(), y_colors):
            tick.set_color(color)

        for y in range(rows + 1):
            ax.axhline(y, color='gray', linestyle='--', linewidth=0.5)
        for x in range(cols + 1):
            ax.axvline(x, color='gray', linestyle='--', linewidth=0.5)

        cbar = fig.colorbar(im, ax=ax, fraction=0.05, pad=0.01)
        cbar.ax.tick_params(labelsize=8)

        ax.set_xlabel("DNA sequence (5′→3′)", fontsize=9)
        ax.set_ylabel("Vancomycin residues", fontsize=9)
        ax.set_title(f"{title_prefix} - {category_info['name']} [{category_info['description']}]", fontsize=10)
        
        filename = f"contact_map_{category_key}_{category_info['description'].replace('~', '_to_').replace(' ', '_').replace('(', '').replace(')', '')}.png"
        plt.savefig(os.path.join(four_cat_dir, filename), bbox_inches='tight', dpi=dpi)
        plt.close()
        
        non_zero_count = np.count_nonzero(matrix)
        total_contacts = matrix.shape[0] * matrix.shape[1]
        percentage = (non_zero_count / total_contacts) * 100
        
        print(f"{category_info['name']} ({category_info['description']}):")
        print(f"  Active contacts: {non_zero_count}/{total_contacts} ({percentage:.1f}%)")
        print(f"  Value range: {matrix.min():.3f} - {matrix.max():.3f}")
        print(f"  Scale used: {vmin:.3f} - {vmax:.3f}")
        print(f"  Saved: {filename}")
        print()
    
    return average_results_dir, four_cat_dir, total_avg_dir

def contact_maps_process(traj_file, top_file, ns_per_window=1.0, dt_per_frame=10.0,
                         stride=1, warmup_frames=110, cutoff=0.65,
                         van_selection="segname VAN", dna_selection="segname DNA",
                         out_dir="./contact_maps", title_prefix="",
                         figsize=(16.5, 2.25), dpi=300,
                         cmap="Reds", vmin=0, vmax=1,
                         save_npz=True, save_total=True, return_counts=False,
                         generate_four_category=True):

    plot_dir = os.path.join(out_dir, "plot")
    npy_dir = os.path.join(out_dir, "npy")
    os.makedirs(plot_dir, exist_ok=True)
    os.makedirs(npy_dir, exist_ok=True)

    frames_per_window = int(round(ns_per_window * 1000 / dt_per_frame))
    frames_per_loaded = int(round(frames_per_window / stride))

    print(f"Frames per loaded window: {frames_per_loaded}")

    sample = next(md.iterload(traj_file, top=top_file, stride=stride))
    topology = sample.topology

    print("\n" + "="*70)
    print("USING HEAVY ATOMS ONLY (NO HYDROGENS)")
    print("="*70)
    
    van_heavy_selection = f"({van_selection}) and not element H"
    dna_heavy_selection = f"({dna_selection}) and not element H"
    
    van_indices = topology.select(van_heavy_selection)
    dna_indices = topology.select(dna_heavy_selection)
    
    print(f"Vancomycin heavy atoms: {len(van_indices)}")
    print(f"DNA heavy atoms: {len(dna_indices)}")

    # Get residue IDs
    van_residue_ids = np.array([topology.atom(i).residue.index for i in van_indices])
    dna_residue_ids = np.array([topology.atom(i).residue.index for i in dna_indices])

    # First get unique residue IDs
    van_unique_ids, van_map_original = np.unique(van_residue_ids, return_inverse=True)
    van_labels_original = [topology.residue(i).name for i in van_unique_ids]
    
    print(f"\n[BEFORE DEDUP] Detected {len(van_labels_original)} residues:")
    print(f"  {van_labels_original}")

    # =====================================================
    # FIX: MERGE DUPLICATE RESIDUE NAMES (PT2, PT6, etc.)
    # =====================================================
    
    print(f"\n[DEDUPLICATION] Merging duplicate residue names...")
    
    residue_groups = OrderedDict()
    for idx, (res_id, res_name) in enumerate(zip(van_unique_ids, van_labels_original)):
        if res_name not in residue_groups:
            residue_groups[res_name] = []
        residue_groups[res_name].append(idx)
    
    van_labels = []
    old_to_new_map = {}
    
    new_idx = 0
    for res_name, old_indices in residue_groups.items():
        if len(old_indices) > 1:
            print(f"  ✓ Merging {len(old_indices)} × {res_name}")
        van_labels.append(res_name)
        for old_idx in old_indices:
            old_to_new_map[old_idx] = new_idx
        new_idx += 1
    
    # Update van_map to use merged indices
    van_map = np.array([old_to_new_map[i] for i in van_map_original])
    
    print(f"\n[AFTER DEDUP] Final {len(van_labels)} residues:")
    print(f"  {van_labels}")
    
    # DNA (no deduplication needed)
    dna_unique, dna_map = np.unique(dna_residue_ids, return_inverse=True)

    dna_residues_ordered = []
    for res_idx in sorted(dna_unique):
        res = topology.residue(res_idx)
        dna_residues_ordered.append(res)
    
    dna_bases = [res.name for res in dna_residues_ordered]
    dna_seq = ''.join([{'DA': 'A', 'A': 'A', 'DC': 'C', 'C': 'C',
                        'DG': 'G', 'G': 'G', 'DT': 'T', 'T': 'T'}.get(b, 'N') for b in dna_bases])
    dna_colors = [dna_color_map.get(b, '#7e7e7e') for b in dna_seq]
    
    print(f"\nDNA sequence length: {len(dna_seq)}")
    print(f"DNA sequence: {dna_seq}")
    print(f"DNA residues detected: {len(dna_unique)}")

    dna_colors_regional = []
    for i in range(len(dna_seq)):
        if i <= 22:
            dna_colors_regional.append("red")
        elif i <= 29:
            dna_colors_regional.append("orange")
        elif i <= 45:
            dna_colors_regional.append("gold")
        elif i <= 52:
            dna_colors_regional.append("green")
        else:
            dna_colors_regional.append("blue")

    y_labels = van_labels
    vanco_colors_dict = {
        "GL2": "pink", "GL1": "pink",
        "PT7": "cyan", "PT6": "cyan", "PT5": "cyan", 
        "PT4": "cyan", "PT3": "cyan", "PT2": "cyan", "PT1": "cyan"
    }
    y_colors = [vanco_colors_dict.get(label, "gray") for label in y_labels]
    
    print(f"\nY-axis labels: {y_labels}")
    print(f"Y-axis colors: {y_colors}")

    buffer = []
    buffer_frames = 0
    skipped = warmup_frames

    def retrieve_window(n):
        nonlocal buffer, buffer_frames
        parts = []
        needed = n
        while needed > 0 and buffer:
            chunk = buffer[0]
            if chunk.n_frames <= needed:
                parts.append(chunk)
                buffer.pop(0)
                buffer_frames -= chunk.n_frames
                needed -= chunk.n_frames
            else:
                parts.append(chunk[:needed])
                buffer[0] = chunk[needed:]
                buffer_frames -= needed
                needed = 0
        if not parts:
            return None
        if len(parts) == 1:
            return parts[0]
        try:
            return md.join(parts)
        except Exception:
            combined = parts[0]
            for p in parts[1:]:
                combined = combined.join(p)
            return combined

    accum_counts = None
    total_frames = 0
    window_idx = 0

    for chunk in md.iterload(traj_file, top=top_file, stride=stride):
        if skipped > 0:
            if chunk.n_frames <= skipped:
                skipped -= chunk.n_frames
                continue
            else:
                chunk = chunk[skipped:]
                skipped = 0
        if chunk.n_frames == 0:
            continue
        buffer.append(chunk)
        buffer_frames += chunk.n_frames

        while buffer_frames >= frames_per_loaded:
            window = retrieve_window(frames_per_loaded)
            if window is None or window.n_frames == 0:
                break

            actual_ns = window.n_frames * dt_per_frame * stride / 1000
            print(f"Window {window_idx + 1} | Frames: {window.n_frames} | Duration ~{actual_ns:.2f} ns")

            freq_matrix = gpu_contact_frequency(
                window, cutoff, van_indices, dna_indices, van_map, dna_map, len(van_labels), len(dna_unique))

            print(f"Contact matrix shape: {freq_matrix.shape}")
            print(f"Expected shape: ({len(van_labels)}, {len(dna_unique)})")
            print(f"Max contact frequency in window: {freq_matrix.max()}")

            if accum_counts is None:
                accum_counts = np.zeros_like(freq_matrix, dtype=np.float64)

            if accum_counts.shape != freq_matrix.shape:
                print("WARNING: Shape mismatch detected, adjusting size...")
                min_rows = min(accum_counts.shape[0], freq_matrix.shape[0])
                min_cols = min(accum_counts.shape[1], freq_matrix.shape[1])
                freq_matrix = freq_matrix[:min_rows, :min_cols]
                accum_counts = accum_counts[:min_rows, :min_cols]

            rows, cols = freq_matrix.shape

            fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
            
            y0, y1 = 0, rows
            im = ax.imshow(freq_matrix, origin='lower', aspect='auto', 
                          extent=[0, cols, y0, y1], 
                          cmap=cmap, vmin=vmin, vmax=vmax)

            ax.set_xlim(0, cols)
            x_ticks = np.arange(cols) + 0.5
            ax.set_xticks(x_ticks)
            ax.set_xticklabels(list(dna_seq[:cols]), fontsize=6, rotation=0, ha='center')
            for tick, color in zip(ax.get_xticklabels(), dna_colors_regional[:cols]):
                tick.set_color(color)

            ax.set_ylim(y0, y1)
            y_ticks = np.arange(rows) + 0.5
            ax.set_yticks(y_ticks)
            ax.set_yticklabels(y_labels[:rows], fontsize=8)
            for tick, color in zip(ax.get_yticklabels(), y_colors[:rows]):
                tick.set_color(color)

            for y in range(rows + 1):
                ax.axhline(y, color='gray', linestyle='--', linewidth=0.5)
            for x in range(cols + 1):
                ax.axvline(x, color='gray', linestyle='--', linewidth=0.5)

            ax.set_xlabel("DNA sequence (5′→3′)", fontsize=9)
            ax.set_ylabel("Vancomycin residues", fontsize=9)
            ax.set_title(f"{title_prefix} [{window_idx * ns_per_window:.0f} - {(window_idx + 1) * ns_per_window:.0f} ns]", fontsize=10)

            cbar = fig.colorbar(im, ax=ax, fraction=0.05, pad=0.01)
            cbar.ax.tick_params(labelsize=8)

            filename = f"contact_map_{window_idx +1:03d}_{int(window_idx*ns_per_window)}-{int((window_idx+1)*ns_per_window)}ns.png"
            plt.savefig(os.path.join(plot_dir, filename), bbox_inches='tight')
            plt.close()

            if save_npz:
                np.savez_compressed(
                    os.path.join(npy_dir, filename.replace('.png', '.npz')),
                    matrix=freq_matrix,
                    x_labels=np.array(list(dna_seq[:cols])),
                    y_labels=np.array(y_labels[:rows]),
                    t_start_ns=np.array([int(window_idx * ns_per_window)], dtype=int),
                    t_end_ns=np.array([int((window_idx + 1) * ns_per_window)], dtype=int),
                    window_index=np.array([window_idx], dtype=int)
                )

            accum_counts += freq_matrix * window.n_frames
            total_frames += window.n_frames
            window_idx += 1

    if save_total and total_frames > 0:
        avg_matrix = accum_counts / total_frames
        avg_matrix = np.clip(avg_matrix, 0, 1).astype(np.float32)
        rows, cols = avg_matrix.shape

        if generate_four_category:
            print("\n" + "="*70)
            print("GENERATING FOUR CATEGORY CONTACT MAPS IN 'average_results' FOLDER")
            print("="*70)
            average_results_dir, four_cat_dir, total_avg_dir = generate_four_category_maps(
                avg_matrix, van_labels, dna_seq[:cols], dna_colors_regional[:cols], 
                y_labels[:rows], y_colors[:rows], title_prefix, figsize, dpi, out_dir, 
                original_cmap=cmap, original_vmin=vmin, original_vmax=vmax)

        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        
        y0, y1 = 0, rows
        im = ax.imshow(avg_matrix, origin='lower', aspect='auto',
                      extent=[0, cols, y0, y1],
                      cmap=cmap, vmin=vmin, vmax=vmax)

        ax.set_xlim(0, cols)
        x_ticks = np.arange(cols) + 0.5
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(list(dna_seq[:cols]), fontsize=6, rotation=0, ha='center')
        for tick, color in zip(ax.get_xticklabels(), dna_colors_regional[:cols]):
            tick.set_color(color)

        ax.set_ylim(y0, y1)
        y_ticks = np.arange(rows) + 0.5
        ax.set_yticks(y_ticks)
        ax.set_yticklabels(y_labels[:rows], fontsize=8)
        for tick, color in zip(ax.get_yticklabels(), y_colors[:rows]):
            tick.set_color(color)

        for y in range(rows + 1):
            ax.axhline(y, color='gray', linestyle='--', linewidth=0.5)
        for x in range(cols + 1):
            ax.axvline(x, color='gray', linestyle='--', linewidth=0.5)

        ax.set_xlabel("DNA sequence (5′→3′)", fontsize=9)
        ax.set_ylabel("Vancomycin residues", fontsize=9)
        total_ns = window_idx * ns_per_window
        ax.set_title(f"{title_prefix} Total [{total_ns:.0f} ns]", fontsize=10)

        cbar = fig.colorbar(im, ax=ax, fraction=0.05, pad=0.01)
        cbar.ax.tick_params(labelsize=8)

        total_filename = f"contact_map_TOTAL_{int(total_ns)}ns.png"
        
        plt.savefig(os.path.join(plot_dir, total_filename), bbox_inches='tight')
        if generate_four_category:
            plt.savefig(os.path.join(total_avg_dir, total_filename), bbox_inches='tight')
        plt.close()

        if save_npz:
            np.savez_compressed(
                os.path.join(npy_dir, total_filename.replace('.png', '.npz')),
                matrix=avg_matrix,
                x_labels=np.array(list(dna_seq[:cols])),
                y_labels=np.array(y_labels[:rows]),
                total_time=np.array([int(total_ns)], dtype=int),
                total_frames=np.array([total_frames], dtype=int),
            )
            if generate_four_category:
                np.savez_compressed(
                    os.path.join(average_results_dir, total_filename.replace('.png', '.npz')),
                    matrix=avg_matrix,
                    x_labels=np.array(list(dna_seq[:cols])),
                    y_labels=np.array(y_labels[:rows]),
                    total_time=np.array([int(total_ns)], dtype=int),
                    total_frames=np.array([total_frames], dtype=int),
                )

        if generate_four_category:
            print(f"\nFour category analysis completed!")
            print(f"Results saved in: {average_results_dir}")
            print(f"  four_category_maps/ (4 category contact maps)")
            print(f"  total_average/ (overall average map)")
            print(f"  contact_matrix_data.npz (raw data)")

        if return_counts:
            return {"contact_counts": accum_counts, "frames": total_frames}


contact_maps_process(
    traj_file="/kaggle/input/cross-validation-dataset-2/give_partham_data/trajectories/system2/complex-rep1-formmgbsa.dcd",
    top_file="/kaggle/input/cross-validation-dataset-2/give_partham_data/pdb/system2/complex.pdb",
    ns_per_window=1.0,
    dt_per_frame=10.0,
    stride=1,
    warmup_frames=0,
    cutoff=0.45,
    van_selection="segname VAN",
    dna_selection="segname DNA",
    out_dir="/kaggle/working/",
    title_prefix="VAN×DNA",
    cmap="Reds",
    vmin=0.0,
    vmax=1.0,
    save_npz=True,
    save_total=True,
    return_counts=True,
    generate_four_category=True
)


Frames per loaded window: 100

USING HEAVY ATOMS ONLY (NO HYDROGENS)
Vancomycin heavy atoms: 101
DNA heavy atoms: 1349

[BEFORE DEDUP] Detected 12 residues:
  ['PT1', 'PT2', 'PT3', 'PT4', 'PT4', 'PT5', 'PT6', 'PT7', 'GL1', 'GL2', 'PT2', 'PT6']

[DEDUPLICATION] Merging duplicate residue names...
  ✓ Merging 2 × PT2
  ✓ Merging 2 × PT4
  ✓ Merging 2 × PT6

[AFTER DEDUP] Final 9 residues:
  ['PT1', 'PT2', 'PT3', 'PT4', 'PT5', 'PT6', 'PT7', 'GL1', 'GL2']

DNA sequence length: 66
DNA sequence: CTCTCGGGACGACCGAGGGTACCGCAATAGTACTTATTGTTCGCCTATTGTGGGTCGGGTCGTCCC
DNA residues detected: 66

Y-axis labels: ['PT1', 'PT2', 'PT3', 'PT4', 'PT5', 'PT6', 'PT7', 'GL1', 'GL2']
Y-axis colors: ['cyan', 'cyan', 'cyan', 'cyan', 'cyan', 'cyan', 'cyan', 'pink', 'pink']
Window 1 | Frames: 100 | Duration ~1.00 ns
Contact matrix shape: (9, 66)
Expected shape: (9, 66)
Max contact frequency in window: 0.0
Window 2 | Frames: 100 | Duration ~1.00 ns
Contact matrix shape: (9, 66)
Expected shape: (9, 66)
Max contact fr

{'contact_counts': array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
         0.00000000e+00, 3.30000000e+01, 4.93000000e+02, 1.27800000e+03,
         1.78000000e+03, 6.44000000e+02, 9.98400000e+03, 2.16799999e+03,
         0.00000000e+00, 4.40000000e+01, 4.25900000e+03, 1.53000000e+02,
         8.70300000e+03, 1.89900000e+03, 2.00000000e+02, 3.10000000e+01,
         0.00000000e+00, 7.77800000e+03, 8.92900001e+03, 0.00000000e+00,
         3.20000000e+01, 7.00000000e+00, 2.58000000e+02, 8.55000000e+02,
         1.01100000e+03, 5.34000000e+02, 8.90000000e+01, 0.00000000e+00,
         0.00000000e+00, 0.000000

In [3]:
import shutil
import os

# Parent directory containing the folders
output_dir = "/kaggle/working/"

# List of subfolders to include
folders_to_zip = ["npy", "plot","average_results"]

# Temporary folder to collect the folders (optional)
temp_dir = "/kaggle/temp_archive"
os.makedirs(temp_dir, exist_ok=True)

# Copy folders into temp folder
for folder in folders_to_zip:
    src = os.path.join(output_dir, folder)
    dst = os.path.join(temp_dir, folder)
    if os.path.exists(src):
        shutil.copytree(src, dst)
    else:
        print(f"Warning: {src} does not exist and will be skipped.")

# Create the zip archive
archive_path = "/kaggle/working/output_folders_archive_System_2"
shutil.make_archive(base_name=archive_path, format='zip', root_dir=temp_dir)

print("Archive created at:", archive_path + ".zip")

# Optional: clean up the temporary folder
shutil.rmtree(temp_dir)

Archive created at: /kaggle/working/output_folders_archive_System_2.zip
